### Strukturierte Ausgaben

Stellen sicher, dass das Modell stets Antworten generiert, die dem von Ihnen bereitgestellten JSON-Schema entsprechen . Sie müssen sich also keine Sorgen machen, dass das Modell einen erforderlichen Schlüssel auslässt oder einen ungültigen Enum-Wert ausgibt.

Zu den Vorteilen strukturierter Ausgaben gehören:
* Zuverlässige Typsicherheit: Falsch formatierte Antworten müssen weder validiert noch erneut gesendet werden.
* Explizite Ablehnungen: Ablehnungen des sicherheitsbasierten Modells sind nun programmatisch erkennbar.
* Einfachere Eingabeaufforderung: Keine streng formulierten Eingabeaufforderungen mehr nötig, um eine einheitliche Formatierung zu erreichen.

Zusätzlich zur Unterstützung von JSON Schema in der REST-API ermöglichen die OpenAI SDKs für Python und JavaScript auch die einfache Definition von Objektschemata mit Pydantic bzw. Zod . 

Im Folgenden wird gezeigt, wie Informationen aus unstrukturiertem Text extrahiert werden.

In [ ]:
%run ~/work/env.py

In [ ]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(api_key=OPENAI_API_KEY, base_url=AI_BASE_URL)

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

response = client.responses.parse(
    model=AI_MODEL,
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent,
)

event = response.output_parsed
print(event)

---
### Gedankenkette
Sie können das Modell anweisen, eine Antwort strukturiert und schrittweise auszugeben, um den Benutzer durch die Lösung zu führen.

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import json

client = OpenAI(api_key=OPENAI_API_KEY, base_url=AI_BASE_URL)

class Step(BaseModel):
    explanation: str
    output: str

class MathReasoning(BaseModel):
    steps: list[Step]
    final_answer: str

response = client.responses.parse(
    model=AI_MODEL,
    input=[
        {
            "role": "system",
            "content": "You are a helpful math tutor. Guide the user through the solution step by step.",
        },
        {"role": "user", "content": "how can I solve 8x + 7 = -23"},
    ],
    text_format=MathReasoning,
)

math_reasoning = response.output_parsed
print(json.dumps(math_reasoning.model_dump(), indent=2, ensure_ascii=False))

---

### UI-Generierung
Gültiges HTML lässt sich generieren, indem man es als rekursive Datenstrukturen mit Einschränkungen, wie z. B. Aufzählungen, darstellt.

In [ ]:
from enum import Enum
from typing import List

from openai import OpenAI
from pydantic import BaseModel
import json

client = OpenAI(api_key=OPENAI_API_KEY, base_url=AI_BASE_URL)

class UIType(str, Enum):
    div = "div"
    button = "button"
    header = "header"
    section = "section"
    field = "field"
    form = "form"

class Attribute(BaseModel):
    name: str
    value: str

class UI(BaseModel):
    type: UIType
    label: str
    children: List["UI"]
    attributes: List[Attribute]

UI.model_rebuild()  # This is required to enable recursive types

class Response(BaseModel):
    ui: UI

response = client.responses.parse(
    model=AI_MODEL,
    input=[
        {
            "role": "system",
            "content": "You are a UI generator AI. Convert the user input into a UI.",
        },
        {"role": "user", "content": "Make a User Profile Form"},
    ],
    text_format=Response,
)

ui = response.output_parsed
print(json.dumps(ui.model_dump(), indent=2, ensure_ascii=False))


---
### Weitere Beispiele

* [OpenAI API - Structured model outputs](https://developers.openai.com/api/docs/guides/structured-outputs)